# Lab 3 — Structured outputs and guardrails

Two final building blocks, and then you'll have the full OpenAI Agents SDK toolkit:
1. **Structured outputs** — force the agent to answer in an exact shape (clean data, not a paragraph).
2. **Guardrails** — automatic checks that protect your system from bad inputs.

We keep using the FlowOps sales example from Lab 2.

In [ ]:
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from agents import Agent, Runner, trace, input_guardrail, GuardrailFunctionOutput

load_dotenv(override=True)

## Structured outputs

Instead of free text, we describe the shape we want with a **Pydantic** model and pass it as `output_type`. The agent must fill it in. Here: a cold email split into a **subject** and a **body**.

In [ ]:
class ColdEmail(BaseModel):
    subject: str = Field(description="A short, compelling subject line.")
    body: str = Field(description="The body of the cold email.")

writer = Agent(
    name="Structured Email Writer",
    instructions="Write a professional cold email for FlowOps, an AI workflow-automation SaaS.",
    model="gpt-5.4-mini",
    output_type=ColdEmail,
)

with trace("Structured email"):
    result = await Runner.run(writer, "Write a cold email to a prospective customer.")

email = result.final_output
print("SUBJECT:", email.subject)
print("BODY:\n", email.body)

We got back real fields — `email.subject` and `email.body` — not a blob of text. That's what lets us build reliable systems and GUIs on top of an agent's answer.

## Guardrails

A **guardrail** runs automatically and can **stop** an agent if something is wrong. An *input* guardrail checks the user's request before the agent acts.

Example: our sales system should not send emails that contain a specific **personal name** (a privacy rule). We build a small agent that detects names, then wrap it as a guardrail.

In [ ]:
class NameCheck(BaseModel):
    contains_name: bool = Field(description="True if the message includes a person's personal name.")
    name: str = Field(description="The name found, or an empty string.")

guardrail_agent = Agent(
    name="Name Check",
    instructions="Check whether the user's message includes someone's personal name.",
    model="gpt-5.4-mini",
    output_type=NameCheck,
)

@input_guardrail
async def no_personal_names(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    found = result.final_output.contains_name
    return GuardrailFunctionOutput(output_info={"found": result.final_output}, tripwire_triggered=found)

In [ ]:
careful_writer = Agent(
    name="Careful Email Writer",
    instructions="Write a professional cold email for FlowOps.",
    model="gpt-5.4-mini",
    input_guardrails=[no_personal_names],
)

# This one is SAFE — no personal name — so it runs normally.
with trace("Guardrail - safe"):
    result = await Runner.run(careful_writer, "Write a cold email addressed to 'Dear Head of Operations'.")
print(result.final_output)

In [ ]:
# This one INCLUDES a personal name, so the guardrail trips and blocks the run.
from agents import InputGuardrailTripwireTriggered

try:
    with trace("Guardrail - blocked"):
        result = await Runner.run(careful_writer, "Write a cold email addressed to Sara Khan.")
    print(result.final_output)
except InputGuardrailTripwireTriggered:
    print("Blocked by guardrail: the message contained a personal name.")

The guardrail caught the name and stopped the agent before it did anything. That's how you keep an agentic system safe and on-policy.

> Note: Ed's original course also covers using *different LLM providers* here. We're staying **OpenAI-only** for this course, so we've skipped that part on purpose.

## Recap — you now have the full toolkit
Across Labs 1–3 you've learned: **agents, system & user prompts, async, tools, agents-as-tools, handoffs, structured outputs, and guardrails.**

## Next: Lab 4 — your own project
There is **no Lab 4 in this repo yet** — because *you* build it.
1. Pick one of the **four project options** your instructor presents.
2. On paper, **sketch the architecture**: which agents, which tools, any handoffs, any guardrails, what structured outputs.
3. Tell **Codex** your choice and let it build the Lab 4 notebook.
4. **Compare** your sketch to what Codex built — same? different? why?
5. Then have Codex turn it into a **running app with a GUI**.

The goal isn't to write syntax — it's to **think like an architect and judge the AI's output.**